In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [ ]:
movies = pd.read_csv("ml-32m/movies.csv")  
ratings = pd.read_csv("ml-32m/ratings.csv")  

In [ ]:
movieRatingsDF = ratings.merge(movies, on="movieId", how="inner")

movieRatingsDF = movieRatingsDF.dropna(subset=["title", "genres"])
movieRatingsDF = movieRatingsDF[~movieRatingsDF["title"].str.contains("[^\x00-\x7F]+", regex=True)]

In [ ]:
movie_data = movieRatingsDF.drop_duplicates("title")[["movieId", "title", "genres"]]

# TF-IDF Vectorization 
tfidf = TfidfVectorizer(token_pattern=r'[^|]+')
tfidf_matrix = tfidf.fit_transform(movie_data['genres'])

# KNN Model using Cosine Similarity
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(tfidf_matrix)


def recommend_movies(movie_title, movie_df, tfidf_matrix, knn_model, n_recommendations=5):
    movie_idx = movie_df[movie_df["title"].str.contains(movie_title, case=False, na=False)].index
    if len(movie_idx) == 0:
        return f"No match found for '{movie_title}'"
    movie_idx = movie_idx[0]
    
    distances, indices = knn_model.kneighbors(tfidf_matrix[movie_idx], n_neighbors=n_recommendations+1)
    recommendations = movie_df.iloc[indices[0][1:]] 
    return recommendations


In [ ]:
# Example
movie_to_search = "Toy Story"
recommendations = recommend_movies(movie_to_search, movie_data, tfidf_matrix, knn_model)
print(f"Movies similar to '{movie_to_search}':\n")
print(recommendations)

Movies similar to 'Toy Story':

          movieId                                   title  \
1487555    157821                Legend of Lemnear (1987)   
4919820    140533           Gwen, the Book of Sand (1985)   
626289       3687           Light Years (Gandahar) (1988)   
10090883   199498      Blade of the Phantom Master (2004)   
383879     150379  Legend of the Millennium Dragon (2011)   

                                      genres  
1487555   Adventure|Animation|Fantasy|Sci-Fi  
4919820   Adventure|Animation|Fantasy|Sci-Fi  
626289    Adventure|Animation|Fantasy|Sci-Fi  
10090883  Adventure|Animation|Fantasy|Sci-Fi  
383879    Adventure|Animation|Fantasy|Sci-Fi  


In [ ]:
relevant_movies = ["Toy Story 2", "A Bug's Life", "Finding Nemo"]

# predicted by the recommender
recommended_titles = recommendations["title"].tolist()

k = 5
hits = sum([1 for title in recommended_titles if title in relevant_movies])
precision_at_k = hits / k

print(f"Precision@{k}: {precision_at_k:.2f}")


Precision@5: 0.00


In [ ]:
tags = pd.read_csv("ml-32m/tags.csv")
tags['tag'] = tags['tag'].fillna('').astype(str)
tags_grouped = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()

movie_data_with_tags = movie_data.merge(tags_grouped, on='movieId', how='left')
movie_data_with_tags['tag'] = movie_data_with_tags['tag'].fillna('')

movie_data_with_tags['combined_features'] = movie_data_with_tags['genres'] + ' ' + movie_data_with_tags['tag']

# TF-IDF Vectorization
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movie_data_with_tags['combined_features'])

# KNN model
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(tfidf_matrix)


NearestNeighbors(algorithm='brute', metric='cosine')

In [15]:
def recommend_movies(movie_title, movie_df, tfidf_matrix, knn_model, n_recommendations=5):
    movie_idx = movie_df[movie_df["title"].str.contains(movie_title, case=False, na=False)].index
    if len(movie_idx) == 0:
        return f"No match found for '{movie_title}'"
    movie_idx = movie_idx[0]
    
    distances, indices = knn_model.kneighbors(tfidf_matrix[movie_idx], n_neighbors=n_recommendations+1)
    return movie_df.iloc[indices[0][1:]]


In [ ]:
movie_to_search = "Toy Story"
recommendations = recommend_movies(movie_to_search, movie_data, tfidf_matrix, knn_model)
print(f"Movies similar to '{movie_to_search}':\n")
print(recommendations)

Movies similar to 'Toy Story':

          movieId                                              title  \
9245043    288609                         Flying Phantom Ship (1969)   
1487848    222903                                  Human Lost (2019)   
1873212    259923                                    Beanfilm (1976)   
30509847   274529                   Interplanetary Revolution (1924)   
212972     183437  World of Tomorrow Episode Two: The Burden of O...   

                    genres  
9245043   Animation|Sci-Fi  
1487848   Animation|Sci-Fi  
1873212   Animation|Sci-Fi  
30509847  Animation|Sci-Fi  
212972    Animation|Sci-Fi  
